## 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

We take a document (here, an HR policy).  
We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.  
We store those pieces in a searchable database (a "vector store").  
When a user asks a question, we search the database for the most relevant pieces, and hand them to an AI model to generate a final answer.

That's it. No magic — just search + a language model.

What we use in this notebook:

- 📄 Data: data/hr_policies.txt (a sample HR policy document)
- 🔢 Embeddings: Jina AI
- 🗄️ Vector store: FAISS
- 🧠 LLM: Groq (fast + free-tier friendly)
- 🕸️ Framework: LangChain (create_agent).

Before running: make sure the notebook's kernel is set to the ragenv virtual environment (top-right corner of VS Code / Jupyter), and that a .env file with GROQ_API_KEY and JINA_API_KEY exists in this folder.


### Step 1 — Import everything we need


We import all the tools upfront so it's clear what's being used and where it comes from.


In [ ]:
import os
from dotenv import load_dotenv

# langchain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings

In [2]:
load_dotenv()


True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

if not groq_key:
    raise ValueError("GROQ_API_KEY is missing")

if not jina_key:
    raise ValueError("JINA_API_KEY is missing")

print("ENV VARS LOADED")


ENV VARS LOADED


LOADING OUR DATA


In [4]:
DATA_FILE_PATH = os.path.join("data" , "hr_policies.txt")

### DATA INGESTION


In [5]:
with open(DATA_FILE_PATH, "r", encoding="utf-8") as f:
    text = f.read()

documents = [
    Document(
        page_content=text,
        metadata={"source": DATA_FILE_PATH}
    )
]

print(f"Loaded file: {DATA_FILE_PATH}")
print(documents)
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data/hr_policies.txt
[Document(metadata={'source': 'data/hr_policies.txt'}, page_content='COMPANY HR POLICY\nVertexon Solutions\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 18 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 7 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 8 days.\nSick leave is separate from annual leave, and employees get 12 paid sick days per year.\nA medical certificate is required for sick leave longer than 3 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 3 days per week, subject to manager approval.\nFully remote work arrangements require written approval from both the department head\nand HR, and are reviewed every 6 months.\nEmployees working from home must be reachable during core hours: 11 AM to 5 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 6 month

##### LANGCHAIN DOCUMENT

Langchain processes everything in form of documents

DOCUMENTS :

PAGE CONTENT -- the actual data

METADATA - extra information about the data (Information about the data)


In [6]:
len(documents)

1

In [7]:
print(documents[0].page_content)

COMPANY HR POLICY
Vertexon Solutions

1. LEAVE POLICY
All full-time employees are entitled to 18 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 7 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 8 days.
Sick leave is separate from annual leave, and employees get 12 paid sick days per year.
A medical certificate is required for sick leave longer than 3 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 3 days per week, subject to manager approval.
Fully remote work arrangements require written approval from both the department head
and HR, and are reviewed every 6 months.
Employees working from home must be reachable during core hours: 11 AM to 5 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 6 months from their date of joining.
During probation, employees accrue leave at half the standard rate, and may take
unpaid le

In [8]:
print(documents[0].metadata)

{'source': 'data/hr_policies.txt'}


#### SPLITTING OUR DATA


In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_chunks = text_splitter.split_documents(documents)
print(f"Number of text chunks created: {len(text_chunks)}")
print(text_chunks)


Number of text chunks created: 8
[Document(metadata={'source': 'data/hr_policies.txt'}, page_content='COMPANY HR POLICY\nVertexon Solutions\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 18 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 7 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 8 days.\nSick leave is separate from annual leave, and employees get 12 paid sick days per year.\nA medical certificate is required for sick leave longer than 3 consecutive days.'), Document(metadata={'source': 'data/hr_policies.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 3 days per week, subject to manager approval.\nFully remote work arrangements require written approval from both the department head\nand HR, and are reviewed every 6 months.\nEmployees working from home must be reachable during core hours: 11 AM to 5 PM.'), Document(

NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata


In [10]:
print(text_chunks[7])

page_content='8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, Admin, and HR departments before their last working day.
An exit interview is mandatory for all departing employees.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 30 days of the last working day.' metadata={'source': 'data/hr_policies.txt'}


In [11]:
print(text_chunks[2].page_content)

3. PROBATION PERIOD
All new employees undergo a probation period of 6 months from their date of joining.
During probation, employees accrue leave at half the standard rate, and may take
unpaid leave in case of emergencies, subject to manager approval.
Performance is reviewed at the 3-month and 6-month marks to confirm employment.


#### EMBEDD OUR DATA


In [12]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en") # type: ignore

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


#### STORE DATA IN VECTOR DB


In [13]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(text_chunks, embeddings_model)
print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 8


In [16]:
test_query = "What is the company's policy on remote work?"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
   

Query: What is the company's policy on remote work?

--- Match 1 ---
2. WORK FROM HOME POLICY
Employees may work from home up to 3 days per week, subject to manager approval.
Fully remote work arrangements require written approval from both the department head
and HR, and are reviewed every 6 months.
Employees working from home must be reachable during core hours: 11 AM to 5 PM.
--- Match 2 ---
5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, internet bills, and co-working space fees used for official work.
All reimbursement claims must be submitted with valid bills within 15 days of the expense.
Claims are processed within 5 working days after approval from the reporting manager.


#### TOOL


In [32]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIVAL


In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(model_name="openai/gpt-oss-120b",temperature=0) # type: ignore

llm.model_name

'openai/gpt-oss-120b'

In [ ]:
response = llm.invoke("Hey is learning RAG hard?")
print(response)

content='Hey!\u202fGreat question. Whether learning Retrieval‑Augmented Generation (RAG) feels “hard” really depends on a few things—your background, the depth you want to go into, and the resources you use. Here’s a quick rundown to help you gauge the learning curve and give you a roadmap if you decide to dive in.\n\n---\n\n## 1️⃣ What is RAG, in a nutshell?\n\n- **Retrieval‑Augmented Generation** combines two stages:\n  1. **Retrieval** – a search over an external knowledge source (documents, vector DB, API, etc.) to pull relevant chunks of information.\n  2. **Generation** – a language model (LLM) that takes the retrieved context plus the user query and produces a response.\n\n- The idea: give the model *up‑to‑date* or *domain‑specific* facts without having to fine‑tune the entire model on massive data.\n\n---\n\n## 2️⃣ How “hard” is it? (Factors)\n\n| Factor | Why it matters | Typical difficulty level |\n|--------|----------------|--------------------------|\n| **Programming experi

In [ ]:
response.content

'Hey!\u202fGreat question. Whether learning Retrieval‑Augmented Generation (RAG) feels “hard” really depends on a few things—your background, the depth you want to go into, and the resources you use. Here’s a quick rundown to help you gauge the learning curve and give you a roadmap if you decide to dive in.\n\n---\n\n## 1️⃣ What is RAG, in a nutshell?\n\n- **Retrieval‑Augmented Generation** combines two stages:\n  1. **Retrieval** – a search over an external knowledge source (documents, vector DB, API, etc.) to pull relevant chunks of information.\n  2. **Generation** – a language model (LLM) that takes the retrieved context plus the user query and produces a response.\n\n- The idea: give the model *up‑to‑date* or *domain‑specific* facts without having to fine‑tune the entire model on massive data.\n\n---\n\n## 2️⃣ How “hard” is it? (Factors)\n\n| Factor | Why it matters | Typical difficulty level |\n|--------|----------------|--------------------------|\n| **Programming experience** |

##### AI AGENT

3 Parts

**LLM** - BRAIN

**TOOL** - SUPER POWER

**MEMORY** - no memory


In [15]:
from langchain.agents import create_agent

print("create_agent: OK")


create_agent: OK


In [33]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Vertexon Solutions. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
    # system_prompt = """ You are a friendly HR assistant working for Vertexon Solutions.
    # Answer general questions clearly and helpfully.
    # Do not invent information about company HR policies.
    # """
) 

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [34]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [35]:
response = hr_assistant.invoke({"messages": [{"role": "user", "content": "Who are you?"}]})

In [36]:
print(response["messages"][-1].content)

I’m your friendly HR assistant here at Vertexon Solutions, ready to help you with any questions you have about our policies, procedures, or anything else related to the workplace. How can I assist you today?


SYSTEM MESSAGE - HR ASSISNT

HUMAN MESSAGE - TELL ME ABOUT POLICIES

AI MESSAGE - HEY THESE ARE THEPLOICES


In [37]:
response = hr_assistant.invoke({"messages": [{"role": "user", "content": "What is the company's policy on remote work?"}]})

In [38]:
print(response["messages"][-1].content)

**Vertexon Solutions – Remote Work (Work‑From‑Home) Policy**

- **Partial remote work:**  
  - Employees may work from home **up to 3 days per week**.  
  - This arrangement requires **approval from your manager**.

- **Fully remote work:**  
  - A completely remote setup is allowed only with **written approval from both the department head and HR**.  
  - These fully remote arrangements are **reviewed every 6 months** to ensure they remain appropriate.

- **Core‑hours availability:**  
  - While working remotely, you must be reachable during the company’s core hours of **11 AM – 5 PM**.

If you’d like to set up a remote‑work schedule, start by discussing it with your manager and, for full‑remote requests, submit the written approval to HR.


In [41]:
response = hr_assistant.invoke({"messages": [{"role": "user", "content": "Tell me which company you work for?"}]})
print(response["messages"][-1].content)

I’m an HR assistant for **Vertexon Solutions**.
